In [211]:
import torch
import torch.nn as nn

In [212]:
from transformer_block import DecoderBlock, EncoderBlock, TransformerBlockConfig

# FSQ

## FSQ

In [213]:
class FSQ:
    def __init__(self, config):
        self.L = torch.as_tensor(config.fsq_levels, dtype=torch.int64, device=config.device) # []
        self.S = (self.L - 1) // 2
        self.weights = torch.as_tensor(
            list(accumulate([1] + config.fsq_levels[:-1], func=operator.mul)),
            dtype=torch.int64,
            device=config.device
        )

    def fsq_quantize(self, z):
        z_scaled = self.S * torch.tanh(z)
        z_round = torch.round(z_scaled)
        z_q = z + (z_round - z).detach()
        return z_q

    def pack_ids(self, q_int):
        q_shift = q_int + self.S
        ids = (self.weights * q_shift).sum(dim=-1)
        return ids

    def unpack_ids(self, ids):
        B, N = ids.shape
        D = self.L.shape[0]

        q_shift = torch.zeros(B, N, D, dtype=torch.int64, device=ids.device)
        remainder = ids.clone()
        for i in range(D - 1, -1, -1):
            q_shift[:,:,i] = remainder // self.weights[i]
            remainder = remainder % self.weights[i]

        return q_shift - self.S

# OAT Tokenizer

## OATConfig

In [214]:
import einops
from dataclasses import dataclass

@dataclass
class OATConfig:
    # ==================== 必填参数 ====================
    action_horizon: int
    d_action: int
    fsq_levels: list

    # ==================== 可选参数 ====================
    d_model: int = 4
    n_heads: int = 2
    dropout_rate: float = 0.1
    num_register_tokens: int = 8
    num_encoder_layers: int = 2
    num_decoder_layers: int = 4
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    @property
    def d_hidden(self):
        return self.d_model * 4

    def __post_init__(self):
        assert len(self.fsq_levels) == self.d_model, "fsq_levels 长度必须等于 d_model"


## OATTokenizer

In [215]:
class OATTokenizer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        transformer_config = TransformerBlockConfig(
                d_model=config.d_model,
                n_heads=config.n_heads,
                d_hidden=config.d_hidden,
                dropout_rate=config.dropout_rate,
        )

        # 1. 动作投影
        self.action_proj = nn.Linear(config.d_action, config.d_model)

        # 2. Registers
        self.registers = nn.Parameter(torch.randn(1, config.num_register_tokens, config.d_model))

        # 3. Mask tokens
        self.mask_tokens = nn.Parameter(torch.randn(1, config.d_model))

        # 4. 编码器
        self.encode_layers = nn.ModuleList(
            EncoderBlock(transformer_config) for _ in range(config.num_encoder_layers)
        )

        # 5. FSQ
        self.fsq = FSQ(config)

        # 6. 解码器
        self.decode_layers = nn.ModuleList(
            DecoderBlock(transformer_config) for _ in range(config.num_decoder_layers)
        )
        self.positions = nn.Parameter(torch.randn(1, config.action_horizon, config.d_model))

        # 7. 输出
        self.fc_out = nn.Linear(config.d_model, config.d_action)


    @property
    def encode_mask(self):
        """
        构建 OAT 专用 attention mask:
          - action tokens 之间：全连接（无 mask）
          - register tokens：causal（只能看左边）
          - register可 attend 所有action tokens
          - action tokens 不能 attend 任何 register tokens
        """
        total_len = self.config.num_register_tokens + self.config.action_horizon
        # 初始化 mask，全为 False: True被屏蔽，False连接
        mask = torch.zeros(total_len, total_len, dtype=torch.bool)

        # register部分：causal mask，上三角表示未来，未来被屏蔽，不包括对角线
        register_causal_mask = torch.triu(torch.ones(self.config.num_register_tokens, self.config.num_register_tokens, dtype=torch.bool),
                                          diagonal=1)

        mask[self.config.action_horizon:, self.config.action_horizon:] = register_causal_mask

        # action 不可见 register
        mask[:self.config.action_horizon, self.config.action_horizon:] = True

        return mask


    def _actions_reconstruct(self, fsq_registers):
        B = fsq_registers.shape[0]

        # Decode
        # 初始化 decode_outputs
        decode_outputs = einops.repeat(
            self.positions,
            '1 n d -> b n d',
            b = B,
        ).to(self.config.device)

        for decode_layer in self.decode_layers:
            decode_outputs = decode_layer(decode_outputs, to_cross_attn=fsq_registers)

        # 重建动作
        return self.fc_out(decode_outputs)

    def fit(self, actions, optimizer):
        # actions [B, H_a, D]
        B = actions.shape[0]

        fsq_registers = self.encode(actions, to_ids=False)
        fsq_registers = fsq_registers.to(self.config.device)

        if self.training:
            # Nested Dropout
            # 随机保留 K 个
            K = torch.randint(1, self.config.num_register_tokens + 1, size=(B, 1), device=self.config.device) # [B, 1]
            mask = torch.arange(1, self.config.num_register_tokens + 1, device=self.config.device).unsqueeze(0) > K # [1, N]

            # mask 会把一整行变成 True or False, True 用 mask_tokens 的部分替代，即整行替换
            fsq_registers = torch.where(mask.unsqueeze(-1),
                                        self.mask_tokens,
                                        fsq_registers)

        # action reconstruct
        actions_recon = self._actions_reconstruct(fsq_registers)

        if hasattr(optimizer, 'train'):
            optimizer.train()

        # 计算损失
        loss = F.mse_loss(actions_recon, actions)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        return loss.item()

    def encode(self, actions, to_ids=True):
        actions = self.action_proj(actions) # [B, T, d_model]
        B, T, D = actions.shape

        registers = einops.repeat(
            self.registers,
            '1 n d -> b n d',
            b = B,
        )

        # Encode
        encode_outputs = torch.cat([actions, registers], dim=1)
        encode_mask = self.encode_mask.to(self.config.device)

        for encode_layer in self.encode_layers:
            encode_outputs = encode_layer(encode_outputs, mask=encode_mask)

        encoded_registers = encode_outputs[:, -self.config.num_register_tokens:]

        # Quantization
        fsq_registers = self.fsq.fsq_quantize(encoded_registers) # [B, N, D]

        if to_ids:
            register_tokens = self.fsq.pack_ids(fsq_registers)
            return register_tokens # [B, N]

        return fsq_registers

    def decode(self, register_tokens, prefix_len):
        B, N = register_tokens.shape

        prefix_len = torch.as_tensor(prefix_len, dtype=torch.int64, device=self.config.device)

        register_tokens = register_tokens.to(self.config.device)
        fsq_registers = self.fsq.unpack_ids(register_tokens) # [B, N] -> [B, N, D]

        # 前缀解码
        mask = torch.arange(1, N + 1, device=self.config.device).unsqueeze(0) > prefix_len.expand(B, 1) # [B, N]
        fsq_registers = torch.where(mask.unsqueeze(-1),
                                    self.mask_tokens,
                                    fsq_registers)

        return self._actions_reconstruct(fsq_registers)

# 模拟数据测试

In [216]:
import torch
import torch.nn.functional as F
from dataclasses import dataclass

# ========== 你的 OATConfig ==========
@dataclass
class OATConfig:
    action_horizon: int
    d_action: int
    fsq_levels: list
    d_model: int = 4
    n_heads: int = 2
    dropout_rate: float = 0.1
    num_register_tokens: int = 8
    num_encoder_layers: int = 2
    num_decoder_layers: int = 4
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    @property
    def d_hidden(self):
        return self.d_model * 4

# ========== 测试函数 ==========

def test_fsq():
    """测试 FSQ 量化、打包、解包"""
    print("=" * 50)
    print("测试 FSQ")
    print("=" * 50)

    config = OATConfig(
        action_horizon=32,
        d_action=7,
        fsq_levels=[8, 5, 5, 5]
    )
    fsq = FSQ(config)

    B, N, D = 4, config.num_register_tokens, config.d_model
    z = torch.randn(B, N, D, device=config.device) * 2

    # 1. 量化
    z_q = fsq.fsq_quantize(z)
    print(f"量化前形状: {z.shape}")
    print(f"量化后形状: {z_q.shape}")

    # 2. 打包
    q_int = torch.round(z_q).long()
    token_ids = fsq.pack_ids(q_int)
    print(f"Token IDs 形状: {token_ids.shape}")

    # 3. 解包
    q_int_recovered = fsq.unpack_ids(token_ids)
    print(f"解包后形状: {q_int_recovered.shape}")

    # 4. 验证
    diff = (q_int - q_int_recovered).abs().max()
    print(f"打包/解包误差: {diff.item()} (应为 0)")
    assert diff == 0, "FSQ 打包/解包失败"

    print("\n✅ FSQ 测试通过\n")


def test_oat_fit():
    """测试 OAT 训练（fit 方法）"""
    print("=" * 50)
    print("测试 OAT fit")
    print("=" * 50)

    config = OATConfig(
        action_horizon=32,
        d_action=7,
        fsq_levels=[8, 5, 5, 5]
    )
    tokenizer = OATTokenizer(config)
    tokenizer.to(config.device)

    optimizer = torch.optim.Adam(tokenizer.parameters(), lr=1e-3)

    B = 4
    actions = torch.randn(B, config.action_horizon, config.d_action, device=config.device)

    losses = []
    for step in range(30):
        loss = tokenizer.fit(actions, optimizer)
        losses.append(loss)
        if (step + 1) % 10 == 0:
            print(f"Step {step+1}, loss: {loss:.4f}")

    print(f"Loss 变化: {losses[0]:.4f} -> {losses[-1]:.4f}")
    assert losses[-1] < losses[0], "Loss 应该下降"
    print("\n✅ OAT fit 测试通过\n")


def test_oat_encode_decode():
    """测试 OAT 的 encode 和 decode"""
    print("=" * 50)
    print("测试 OAT encode/decode")
    print("=" * 50)

    config = OATConfig(
        action_horizon=32,
        d_action=7,
        fsq_levels=[8, 5, 5, 5]
    )
    tokenizer = OATTokenizer(config)
    tokenizer.to(config.device)

    # 先训练一下
    optimizer = torch.optim.Adam(tokenizer.parameters(), lr=1e-3)
    B = 4
    actions = torch.randn(B, config.action_horizon, config.d_action, device=config.device)

    for _ in range(50):
        tokenizer.fit(actions, optimizer)

    # 测试 encode/decode
    tokenizer.eval()
    with torch.no_grad():
        # 编码
        token_ids = tokenizer.encode(actions)
        print(f"Token IDs 形状: {token_ids.shape}")
        print(f"Token IDs 范围: [{token_ids.min()}, {token_ids.max()}]")

        # 完整解码
        actions_recon_full = tokenizer.decode(token_ids, prefix_len=config.num_register_tokens)
        print(f"完整解码形状: {actions_recon_full.shape}")

        # 前缀解码（只用前 4 个 token）
        actions_recon_partial = tokenizer.decode(token_ids, prefix_len=4)
        print(f"前缀解码形状 (K=4): {actions_recon_partial.shape}")

        # 计算误差
        mse_full = F.mse_loss(actions_recon_full, actions)
        mse_partial = F.mse_loss(actions_recon_partial, actions)
        print(f"完整解码 MSE: {mse_full:.6f}")
        print(f"前缀解码 MSE (K=4): {mse_partial:.6f}")

        # 前缀解码误差应该更大
        assert mse_partial > mse_full, "前缀解码误差应该大于完整解码"

    print("\n✅ OAT encode/decode 测试通过\n")


if __name__ == "__main__":
    test_fsq()
    test_oat_fit()
    test_oat_encode_decode()

    print("=" * 50)
    print("🎉 所有测试通过！")
    print("=" * 50)

测试 FSQ
量化前形状: torch.Size([4, 8, 4])
量化后形状: torch.Size([4, 8, 4])
Token IDs 形状: torch.Size([4, 8])
解包后形状: torch.Size([4, 8, 4])
打包/解包误差: 0 (应为 0)

✅ FSQ 测试通过

测试 OAT fit
Step 10, loss: 3.9793
Step 20, loss: 2.6337
Step 30, loss: 1.9721
Loss 变化: 6.4715 -> 1.9721

✅ OAT fit 测试通过

测试 OAT encode/decode
Token IDs 形状: torch.Size([4, 8])
Token IDs 范围: [30.0, 995.0]
完整解码形状: torch.Size([4, 32, 7])
前缀解码形状 (K=4): torch.Size([4, 32, 7])
完整解码 MSE: 1.243807
前缀解码 MSE (K=4): 1.279301

✅ OAT encode/decode 测试通过

🎉 所有测试通过！


# 跳转至结尾